# Paso 8: ablación completa y reproducible

Este cuaderno vuelve a correr **todas** las configuraciones en una sola sesión, para que los números del artículo sean internamente consistentes y comparables entre sí.

Qué cambia respecto a los cuadernos anteriores:

- Se activa `enable_op_determinism()`, de modo que dos ejecuciones con la misma semilla den el mismo resultado. Esto elimina la discrepancia entre las corridas de la ventana extendida de los pasos 6 y 7.
- Cada configuración se entrena **tres veces**, incluida la del ajuste de umbral, que antes tenía una sola ejecución.
- El umbral se busca con la **misma rejilla** para todas las configuraciones, ponderando las clases S y V.
- Las versiones con y sin umbral comparten los mismos modelos entrenados, así que están pareadas.
- Se reporta el F1 macro de cuatro y de tres clases, el número de parámetros y el tiempo por latido.

**Antes de ejecutar:** activa la **T4 GPU**. Tarda entre 1,5 y 2 horas (el determinismo hace el entrenamiento algo más lento). Si tienes prisa, pon `INCLUIR_AUMENTO = False` en la sección 8 y baja a una hora.

## 1. Instalar, activar determinismo y descargar datos

In [ ]:
!pip install wfdb -q
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import wfdb, time
import numpy as np, pandas as pd
import tensorflow as tf
from scipy.signal import lfilter
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Hace reproducibles las operaciones de GPU. Si alguna capa no tuviera
# implementacion determinista, este bloque lo avisa y el cuaderno sigue.
DETERMINISTA = True
try:
    tf.config.experimental.enable_op_determinism()
    print('Determinismo activado')
except Exception as e:
    DETERMINISTA = False
    print('No se pudo activar el determinismo:', e)

wfdb.dl_database('mitdb', dl_dir='mitdb')
print('Descarga lista')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.7 MB/s eta 0:00:00
Determinismo activado
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Ge

## 2. Preparar los latidos

In [ ]:
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
AAMI = {'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S','V':'V','E':'V','F':'F'}
CODIGOS_LATIDO = set('NLRejAaJSVEF/fQ')
CLASES = ['N','S','V','F']
FS = 360
ANTES, DESPUES = 324, 162     # ventana extendida: 486 muestras
CORTE = ANTES - 90            # para recortar la ventana base de 252 muestras
SEMILLAS = [0, 1, 2]

def filtro_pasa_altas(x, fs=FS, fc=1.0):
    alpha = np.exp(-2*np.pi*fc/fs)
    return x - lfilter([1-alpha], [1, -alpha], x)

def extraer_latidos(num):
    ruta = f'mitdb/{num}'
    rec = wfdb.rdrecord(ruta); ann = wfdb.rdann(ruta, 'atr')
    canal = rec.sig_name.index('MLII') if 'MLII' in rec.sig_name else 0
    x = filtro_pasa_altas(rec.p_signal[:, canal]); x = (x - x.mean())/x.std()
    pos = np.array([p for p, s in zip(ann.sample, ann.symbol) if s in CODIGOS_LATIDO])
    sim = [s for s in ann.symbol if s in CODIGOS_LATIDO]
    rr = np.diff(pos)/FS; rr_medio = rr.mean()
    V, R, Y = [], [], []
    for i in range(1, len(pos)-1):
        if sim[i] not in AAMI: continue
        r = pos[i]
        if r-ANTES < 0 or r+DESPUES > len(x): continue
        V.append(x[r-ANTES : r+DESPUES])
        R.append([rr[i-1], rr[i], rr[max(0,i-10):i].mean(), rr[i-1]/rr_medio, rr[i]/rr_medio])
        Y.append(AAMI[sim[i]])
    return V, R, Y

def construir(lista):
    V, R, Y = [], [], []
    for n in lista:
        v, r, y = extraer_latidos(n); V += v; R += r; Y += y
    return np.array(V, dtype='float32'), np.array(R, dtype='float32'), np.array(Y)

V_ent, R_ent, y_ent = construir(DS1)
V_pru, R_pru, y_pru = construir(DS2)
a_numero = {c:i for i,c in enumerate(CLASES)}
y_ent_num = np.array([a_numero[c] for c in y_ent])
y_pru_num = np.array([a_numero[c] for c in y_pru])

V_tr, V_val, R_tr, R_val, y_tr, y_val = train_test_split(
    V_ent, R_ent, y_ent_num, test_size=0.1, stratify=y_ent_num, random_state=42)

def recortar(V):
    return V[:, CORTE:CORTE+252]

print('Entrenamiento:', len(y_tr), '| validacion:', len(y_val), '| prueba:', len(y_pru))
print('Latidos por clase en DS1:', {c: int((y_ent_num==a_numero[c]).sum()) for c in CLASES})
print('Latidos por clase en DS2:', {c: int((y_pru_num==a_numero[c]).sum()) for c in CLASES})

Entrenamiento: 45868 | validacion: 5097 | prueba: 49658
Latidos por clase en DS1: {'N': 45822, 'S': 943, 'V': 3786, 'F': 414}
Latidos por clase en DS2: {'N': 44215, 'S': 1836, 'V': 3219, 'F': 388}


## 3. Aumento de datos para las clases minoritarias

In [ ]:
rng_aug = np.random.default_rng(0)

def aumentar(V, R, y, clases=('S','F'), objetivo=5000):
    Vs, Rs, ys = [V], [R], [y]
    for c in clases:
        i = a_numero[c]
        idx = np.where(y == i)[0]
        faltan = max(0, objetivo - len(idx))
        if faltan == 0: continue
        elegidos = rng_aug.choice(idx, faltan, replace=True)
        base = V[elegidos].copy()
        desp = rng_aug.integers(-10, 11, len(elegidos))
        base = np.stack([np.roll(b, d) for b, d in zip(base, desp)])
        base *= rng_aug.uniform(0.85, 1.15, (len(elegidos), 1)).astype('float32')
        base += rng_aug.normal(0, 0.02, base.shape).astype('float32')
        rr = R[elegidos] * rng_aug.uniform(0.98, 1.02, (len(elegidos), 5)).astype('float32')
        Vs.append(base.astype('float32')); Rs.append(rr.astype('float32'))
        ys.append(np.full(len(elegidos), i))
    V2, R2, y2 = np.vstack(Vs), np.vstack(Rs), np.concatenate(ys)
    orden = rng_aug.permutation(len(y2))
    return V2[orden], R2[orden], y2[orden]

V_aug, R_aug, y_aug = aumentar(V_tr, R_tr, y_tr)
print('Entrenamiento aumentado:', {c: int((y_aug==a_numero[c]).sum()) for c in CLASES})

Entrenamiento aumentado: {'N': 41239, 'S': 5000, 'V': 3407, 'F': 5000}


## 4. Modelo y entrenamiento

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def armar_cnn(largo):
    ef = keras.Input(shape=(largo, 1))
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(ef)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    er = keras.Input(shape=(5,))
    z = layers.concatenate([x, er])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    s = layers.Dense(4, activation='softmax')(z)
    m = keras.Model([ef, er], s)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def entrenar(Vtr, Rtr, ytr, Vval, Rval, yval, semilla):
    keras.utils.set_random_seed(semilla)
    m = armar_cnn(Vtr.shape[1])
    pesos = {i: w for i, w in enumerate(
        compute_class_weight('balanced', classes=np.arange(4), y=ytr))}
    m.fit([Vtr[..., None], Rtr], ytr,
          validation_data=([Vval[..., None], Rval], yval),
          epochs=40, batch_size=256, class_weight=pesos,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)],
          verbose=0)
    return m

## 5. Búsqueda de umbral y registro de métricas

Cada llamada a `correr` entrena una vez por semilla y evalúa dos configuraciones con esos mismos modelos: sin ajuste y con los pesos de decisión elegidos en validación.

In [ ]:
# Los pesos de decision se buscan SOLO en validacion, con la misma rejilla
# para todas las configuraciones.
REJILLA_S = np.round(np.arange(0.2, 3.01, 0.2), 2)
REJILLA_V = np.round(np.arange(0.4, 2.01, 0.2), 2)

def buscar_pesos(prob_val):
    mejor, mejor_f1 = (1.0, 1.0), -1.0
    for ws in REJILLA_S:
        for wv in REJILLA_V:
            p = (prob_val * np.array([1.0, ws, wv, 1.0])).argmax(1)
            f1 = f1_score(y_val, p, labels=range(4), average='macro', zero_division=0)
            if f1 > mejor_f1:
                mejor_f1, mejor = f1, (ws, wv)
    return mejor

filas, matrices = [], {}

def registrar(etiqueta, semilla, pred, extra=None):
    se = recall_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    pp = precision_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    f1c = f1_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    fila = {'config': etiqueta, 'semilla': semilla,
            'F1_macro_4': f1c.mean(), 'F1_macro_3': f1c[:3].mean()}
    for i, c in enumerate(CLASES):
        fila[f'Se_{c}'] = se[i]; fila[f'+P_{c}'] = pp[i]
    if extra: fila.update(extra)
    filas.append(fila)
    if semilla == 0:
        matrices[etiqueta] = confusion_matrix(y_pru_num, pred, labels=range(4))
    print(f'  {etiqueta} (semilla {semilla}): F1(4)={fila["F1_macro_4"]:.3f}  '
          f'F1(3)={fila["F1_macro_3"]:.3f}  Se_S={se[1]:.3f}  +P_S={pp[1]:.3f}  '
          f'Se_V={se[2]:.3f}  +P_V={pp[2]:.3f}')

def correr(nombre_sin, nombre_con, Vtr, Rtr, ytr, Vval, Vpru):
    # Entrena una vez por semilla y registra la version sin y con ajuste de umbral.
    for s in SEMILLAS:
        m = entrenar(Vtr, Rtr, ytr, Vval, R_val, y_val, s)
        prob_val = m.predict([Vval[..., None], R_val], verbose=0)
        inicio = time.time()
        prob_pru = m.predict([Vpru[..., None], R_pru], verbose=0)
        ms = (time.time()-inicio)*1000/len(Vpru)
        registrar(nombre_sin, s, prob_pru.argmax(1),
                  {'ms': ms, 'parametros': m.count_params(), 'peso_S': 1.0, 'peso_V': 1.0})
        ws, wv = buscar_pesos(prob_val)
        pred = (prob_pru * np.array([1.0, ws, wv, 1.0])).argmax(1)
        registrar(nombre_con, s, pred,
                  {'ms': ms, 'parametros': m.count_params(), 'peso_S': ws, 'peso_V': wv})

## 6. Ventana base (configuraciones A y D)

In [ ]:
print('--- Ventana base (252 muestras) ---')
correr('A. Ventana base', 'D. Base + umbral',
       recortar(V_tr), R_tr, y_tr, recortar(V_val), recortar(V_pru))

--- Ventana base (252 muestras) ---
  A. Ventana base (semilla 0): F1(4)=0.564  F1(3)=0.751  Se_S=0.585  +P_S=0.295  Se_V=0.942  +P_V=0.880
  D. Base + umbral (semilla 0): F1(4)=0.555  F1(3)=0.740  Se_S=0.350  +P_S=0.347  Se_V=0.962  +P_V=0.859
  A. Ventana base (semilla 1): F1(4)=0.585  F1(3)=0.777  Se_S=0.841  +P_S=0.326  Se_V=0.955  +P_V=0.876
  D. Base + umbral (semilla 1): F1(4)=0.571  F1(3)=0.758  Se_S=0.411  +P_S=0.398  Se_V=0.970  +P_V=0.857
  A. Ventana base (semilla 2): F1(4)=0.557  F1(3)=0.730  Se_S=0.626  +P_S=0.247  Se_V=0.926  +P_V=0.877
  D. Base + umbral (semilla 2): F1(4)=0.529  F1(3)=0.701  Se_S=0.240  +P_S=0.305  Se_V=0.972  +P_V=0.804


## 7. Ventana extendida (configuraciones E y F)

In [ ]:
print('--- Ventana extendida (486 muestras) ---')
correr('E. Ventana extendida', 'F. Extendida + umbral',
       V_tr, R_tr, y_tr, V_val, V_pru)

--- Ventana extendida (486 muestras) ---
  E. Ventana extendida (semilla 0): F1(4)=0.565  F1(3)=0.753  Se_S=0.897  +P_S=0.378  Se_V=0.853  +P_V=0.717
  F. Extendida + umbral (semilla 0): F1(4)=0.606  F1(3)=0.809  Se_S=0.674  +P_S=0.593  Se_V=0.857  +P_V=0.805
  E. Ventana extendida (semilla 1): F1(4)=0.540  F1(3)=0.718  Se_S=0.820  +P_S=0.345  Se_V=0.930  +P_V=0.613
  F. Extendida + umbral (semilla 1): F1(4)=0.573  F1(3)=0.762  Se_S=0.561  +P_S=0.532  Se_V=0.924  +P_V=0.682
  E. Ventana extendida (semilla 2): F1(4)=0.544  F1(3)=0.714  Se_S=0.829  +P_S=0.289  Se_V=0.906  +P_V=0.716
  F. Extendida + umbral (semilla 2): F1(4)=0.538  F1(3)=0.706  Se_S=0.382  +P_S=0.337  Se_V=0.938  +P_V=0.744


## 8. Ventana extendida con aumento de datos (configuraciones C y C+)

In [ ]:
INCLUIR_AUMENTO = True   # ponlo en False si quieres ahorrar unos 30 minutos

if INCLUIR_AUMENTO:
    print('--- Ventana extendida con aumento de datos ---')
    correr('C. Extendida + aumento', 'C+. Aumento + umbral',
           V_aug, R_aug, y_aug, V_val, V_pru)
else:
    print('Omitido')

--- Ventana extendida con aumento de datos ---
  C. Extendida + aumento (semilla 0): F1(4)=0.514  F1(3)=0.684  Se_S=0.902  +P_S=0.331  Se_V=0.925  +P_V=0.536
  C+. Aumento + umbral (semilla 0): F1(4)=0.578  F1(3)=0.770  Se_S=0.845  +P_S=0.507  Se_V=0.908  +P_V=0.660
  C. Extendida + aumento (semilla 1): F1(4)=0.511  F1(3)=0.681  Se_S=0.712  +P_S=0.301  Se_V=0.919  +P_V=0.577
  C+. Aumento + umbral (semilla 1): F1(4)=0.514  F1(3)=0.685  Se_S=0.403  +P_S=0.396  Se_V=0.927  +P_V=0.589
  C. Extendida + aumento (semilla 2): F1(4)=0.487  F1(3)=0.648  Se_S=0.493  +P_S=0.239  Se_V=0.900  +P_V=0.605
  C+. Aumento + umbral (semilla 2): F1(4)=0.465  F1(3)=0.620  Se_S=0.272  +P_S=0.226  Se_V=0.933  +P_V=0.566


## 9. Tabla final

In [ ]:
df = pd.DataFrame(filas)
orden = ['A. Ventana base', 'D. Base + umbral', 'E. Ventana extendida',
         'F. Extendida + umbral', 'C. Extendida + aumento', 'C+. Aumento + umbral']
orden = [o for o in orden if o in set(df['config'])]
cols = ['F1_macro_4','F1_macro_3'] + [f'Se_{c}' for c in CLASES] + [f'+P_{c}' for c in CLASES]
res = df.groupby('config')[cols].agg(['mean','std']).reindex(orden)

tabla = pd.DataFrame(index=res.index)
for c in cols:
    tabla[c] = [f'{a:.3f} ± {b:.3f}' for a, b in zip(res[(c,'mean')], res[(c,'std')])]
tabla['parametros'] = df.groupby('config')['parametros'].first().reindex(orden).values
tabla['ms'] = df.groupby('config')['ms'].mean().reindex(orden).round(3).values

pd.set_option('display.width', 250)
print('ABLACION COMPLETA SOBRE DS2 (promedio de 3 ejecuciones, determinismo activado)')
print(tabla.T)
df.to_csv('ablacion_final.csv', index=False)
tabla.to_csv('ablacion_final_resumen.csv')

print()
print('Pesos de decision elegidos en validacion:')
print(df[df['peso_S'] != 1.0][['config','semilla','peso_S','peso_V']].to_string(index=False))

ABLACION COMPLETA SOBRE DS2 (promedio de 3 ejecuciones, determinismo activado)
config     A. Ventana base D. Base + umbral E. Ventana extendida F. Extendida + umbral C. Extendida + aumento C+. Aumento + umbral
F1_macro_4   0.569 ± 0.015    0.552 ± 0.021        0.550 ± 0.013         0.572 ± 0.034          0.504 ± 0.015        0.519 ± 0.057
F1_macro_3   0.753 ± 0.023    0.733 ± 0.029        0.728 ± 0.022         0.759 ± 0.051          0.671 ± 0.020        0.692 ± 0.076
Se_N         0.911 ± 0.019    0.949 ± 0.010        0.880 ± 0.033         0.925 ± 0.032          0.827 ± 0.019        0.861 ± 0.026
Se_S         0.684 ± 0.138    0.334 ± 0.087        0.849 ± 0.042         0.539 ± 0.147          0.702 ± 0.205        0.507 ± 0.300
Se_V         0.941 ± 0.014    0.968 ± 0.005        0.896 ± 0.039         0.906 ± 0.043          0.914 ± 0.013        0.923 ± 0.013
Se_F         0.033 ± 0.039    0.018 ± 0.014        0.046 ± 0.070         0.054 ± 0.081          0.009 ± 0.001        0.009 ± 0.001
+P_N

## 10. Filas en LaTeX y matrices de confusión

In [ ]:
print('FILAS PARA LA TABLA DE ABLACION (LaTeX)')
print()
for cfg in orden:
    m = res.loc[cfg]
    def v(c):
        return f'{m[(c,"mean")]:.3f} $\\pm$ {m[(c,"std")]:.3f}'.replace('.', ',').replace(',pm', '.pm')
    print(f'{cfg} & {v("F1_macro_4")} & {v("Se_S")} & {v("+P_S")} & '
          f'{v("Se_V")} & {v("+P_V")} & {v("Se_F")} \\\\ \\hline')

print()
print('MATRICES DE CONFUSION (semilla 0)')
for k in orden:
    if k in matrices:
        print()
        print(k)
        print(matrices[k])

FILAS PARA LA TABLA DE ABLACION (LaTeX)

A. Ventana base & 0,569 $\pm$ 0,015 & 0,684 $\pm$ 0,138 & 0,289 $\pm$ 0,040 & 0,941 $\pm$ 0,014 & 0,878 $\pm$ 0,002 & 0,033 $\pm$ 0,039 \\ \hline
D. Base + umbral & 0,552 $\pm$ 0,021 & 0,334 $\pm$ 0,087 & 0,350 $\pm$ 0,046 & 0,968 $\pm$ 0,005 & 0,840 $\pm$ 0,031 & 0,018 $\pm$ 0,014 \\ \hline
E. Ventana extendida & 0,550 $\pm$ 0,013 & 0,849 $\pm$ 0,042 & 0,337 $\pm$ 0,045 & 0,896 $\pm$ 0,039 & 0,682 $\pm$ 0,060 & 0,046 $\pm$ 0,070 \\ \hline
F. Extendida + umbral & 0,572 $\pm$ 0,034 & 0,539 $\pm$ 0,147 & 0,487 $\pm$ 0,134 & 0,906 $\pm$ 0,043 & 0,744 $\pm$ 0,062 & 0,054 $\pm$ 0,081 \\ \hline
C. Extendida + aumento & 0,504 $\pm$ 0,015 & 0,702 $\pm$ 0,205 & 0,291 $\pm$ 0,047 & 0,914 $\pm$ 0,013 & 0,573 $\pm$ 0,035 & 0,009 $\pm$ 0,001 \\ \hline
C+. Aumento + umbral & 0,519 $\pm$ 0,057 & 0,507 $\pm$ 0,300 & 0,376 $\pm$ 0,142 & 0,923 $\pm$ 0,013 & 0,605 $\pm$ 0,049 & 0,009 $\pm$ 0,001 \\ \hline

MATRICES DE CONFUSION (semilla 0)

A. Ventana base
[[41198

## 11. Descargar resultados

In [ ]:
from google.colab import files
files.download('ablacion_final.csv')
files.download('ablacion_final_resumen.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>